In [ ]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_gillette_fifa_2026():
    options = Options()
    # DETACH keeps the window open so you can manually solve a CAPTCHA if it pops up
    options.add_experimental_option("detach", True)
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    # Hide the 'automated' flag from Google's detection
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    try:
        # Constructing the search for exactly the 2026 dates (June 13 is the first Boston match)
        url = "https://www.google.com/travel/search?q=vacation%20rentals%20near%20Gillette%20Stadium%20June%202026&f=ho:1;hl:en-US;"
        driver.get(url)
        
        print("Waiting for properties to appear... If a CAPTCHA appears, solve it now!")
        
        # INCREASED WAIT: Google's travel map can take 15+ seconds to fetch the '25 mile' inventory
        wait = WebDriverWait(driver, 30)
        
        # This selector targets the 'c-wiz' container which holds the hotel cards in 2026
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "c-wiz > div > a")))

        # Scroll to force the 25-mile radius to load
        for _ in range(4):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(2)

        results = []
        # In 2026, the property links are structured inside c-wiz containers
        property_links = driver.find_elements(By.CSS_SELECTOR, "c-wiz > div > a")

        for link_element in property_links:
            try:
                # Find the parent container of the link to get text data
                parent = link_element.find_element(By.XPATH, "..")
                raw_text = parent.text
                lines = raw_text.split('\n')

                name = lines[0] if lines else "N/A"
                price = "N/A"
                miles = "N/A"

                for line in lines:
                    if "$" in line:
                        price = line
                    if "miles" in line.lower():
                        miles = line

                results.append({
                    "airbnb_name": name,
                    "miles_from_Gillette": miles,
                    "price_per_night": price,
                    "guests": "See URL",
                    "bedrooms": "See URL",
                    "url": link_element.get_attribute("href")
                })
            except:
                continue

        if results:
            df = pd.DataFrame(results)
            # Remove duplicates (common in Google Travel scrapes)
            df = df.drop_duplicates(subset=['airbnb_name'])
            df.to_csv("gillette_fifa_stays.csv", index=False)
            print(f"Success! {len(df)} properties captured.")
        else:
            print("Zero properties found. Check if the browser shows a 'Human Verification' screen.")

    except Exception as e:
        print(f"Script stopped: {e}")

if __name__ == "__main__":
    scrape_gillette_fifa_2026()

Waiting for properties to appear... If a CAPTCHA appears, solve it now!
Success! 16 properties captured.


In [13]:
import pandas as pd

# Load the data
df = pd.read_csv("gillette_fifa_stays.csv")

# This ensures you see all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Show the results
print(df)

                                                                          airbnb_name  \
0                                                                         Gaard motel   
1                                     World Cup HQ < 1 Mi to Stadium & Patriot Place!   
2                              Perfect 2-bedroom house with AC in charming Foxborough   
3        Walk path to Gillette stadium/trains. Water access,6 acres. 2units for June.   
4         Location, Location, Location! .9 mile walk to World Cup / Gillette Stadium.   
5     Walk to Gillette! 3BR Home Near Stadium Patriot Place, Drive Boston/ Providence   
6                    Spacious 4-Bedroom Norfolk Home, 20-Min Walk to Gillette Stadium   
7       Friends Getaway! 2 Spacious Units, Free Parking, Minutes to Gillette Stadium!   
8                     Walk to Gillette Stadium / 3BR/3BA / Perfect for World Cup 2026   
9      1.6 Miles From All The Soccer Action. Spacious, Beautiful and Private 5B/3.5Ba   
10                   

In [15]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    # Radius of Earth in miles
    R = 3958.8
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# Gillette Stadium Coords
gillette_lat, gillette_lon = 42.0909, -71.2643

# Apply to your new DF (ensure lat/lon are floats first)
df['calc_miles'] = df.apply(lambda row: haversine(float(row['latitude']), float(row['longitude']), gillette_lat, gillette_lon) if row['latitude'] != "N/A" else None, axis=1)

In [36]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_gillette_fifa_25mile_v2():
    options = Options()
    options.add_experimental_option("detach", True)
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    try:
        # 1. 重新微調過的 25 英里邊界
        map_bounds = "[[41.8900,-71.5500],[42.2900,-70.9700]]"
        
        # 2. 構造 URL
        url = (
            f"https://www.google.com/travel/search?"
            f"q=vacation%20rentals%20in%20Foxborough%20MA&"
            f"hl=en-US&"
            f"f=ho:1;mv:{map_bounds};"
        )
        
        print("Opening Foxborough area with 25-mile bounds...")
        driver.get(url)
        time.sleep(10)

        # 3. 點擊「在此區域搜尋」刷新
        try:
            print("Trying to click 'Search this area'...")
            search_here_btn = driver.find_element(By.XPATH, "//span[contains(text(), 'Search this area')]")
            search_here_btn.click()
            time.sleep(5)
        except:
            print("Already showing the correct area.")

        # 4. 等待列表加載
        wait = WebDriverWait(driver, 20)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='listitem']")))

        print("Scrolling to load more properties...")
        for _ in range(10):
            try:
                container = driver.find_element(By.CSS_SELECTOR, "div[role='main']")
                driver.execute_script("arguments[0].scrollBy(0, 1200);", container)
            except:
                driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(2.5)

        # 5. 抓取數據 - 強化解析邏輯
        results = []
        property_elements = driver.find_elements(By.CSS_SELECTOR, "div[role='listitem']")
        print(f"Detected {len(property_elements)} items. Starting deep parse...")

        for index, card in enumerate(property_elements):
            try:
                # 滾動到該元素確保內容渲染
                if index % 5 == 0:
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", card)
                    time.sleep(0.5)

                # 提取名稱
                try:
                    name = card.find_element(By.TAG_NAME, "h2").text
                except:
                    name = card.get_attribute("aria-label") or "N/

SyntaxError: unterminated string literal (detected at line 75) (1538566497.py, line 75)

In [40]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_gillette_fifa_2026():
    options = Options()
    options.add_experimental_option("detach", True)
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    try:
        # --- 核心修改：定義 25 英里範圍 ---
        # 體育場中心約在 (42.09, -71.26)，這組座標定義了周邊 25 英里的矩形
        map_bounds = "[[41.8500,-71.5500],[42.3000,-70.9500]]"
        
        # 將 mv 參數放入 URL，這會強制 Google 打開大範圍地圖
        url = (
            f"https://www.google.com/travel/search?"
            f"q=vacation%20rentals%20near%20Gillette%20Stadium%20June%202026&"
            f"hl=en-US&"
            f"f=ho:1;mv:{map_bounds};"
        )
        # ------------------------------

        driver.get(url)
        
        print("Waiting for properties to appear... If a CAPTCHA appears, solve it now!")
        
        wait = WebDriverWait(driver, 30)
        
        # 修改點：Google Travel 的房源卡片有時用 listitem 偵測更穩，
        # 為了相容你的 c-wiz 邏輯，我們等到列表出現
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[role='listitem']")))

        # 增加捲動次數：因為範圍變大，房源變多，需要多捲幾次才會加載
        for _ in range(10):
            # 嘗試捲動左側容器
            try:
                container = driver.find_element(By.CSS_SELECTOR, "div[role='main']")
                driver.execute_script("arguments[0].scrollBy(0, 1000);", container)
            except:
                driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(2)

        results = []
        # 保持你的 c-wiz 連結抓取邏輯
        property_links = driver.find_elements(By.CSS_SELECTOR, "c-wiz div a")

        for link_element in property_links:
            try:
                parent = link_element.find_element(By.XPATH, "..")
                raw_text = parent.text
                if not raw_text: continue # 跳過空白內容

                lines = raw_text.split('\n')
                name = lines[0] if lines else "N/A"
                price = "N/A"
                miles = "N/A"

                for line in lines:
                    if "$" in line:
                        price = line
                    if "miles" in line.lower():
                        miles = line

                results.append({
                    "airbnb_name": name,
                    "miles_from_Gillette": miles,
                    "price_per_night": price,
                    "url": link_element.get_attribute("href")
                })
            except:
                continue

        if results:
            df = pd.DataFrame(results)
            df = df.drop_duplicates(subset=['airbnb_name'])
            df.to_csv("gillette_fifa_stays_25mile.csv", index=False)
            print(f"Success! {len(df)} properties captured within the 25-mile range.")
        else:
            print("Zero properties found. Check the browser screen.")

    except Exception as e:
        print(f"Script stopped: {e}")

if __name__ == "__main__":
    scrape_gillette_fifa_2026()

Waiting for properties to appear... If a CAPTCHA appears, solve it now!
Zero properties found. Check the browser screen.


In [41]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

def scrape_gillette_fifa_2026():
    options = Options()
    options.add_experimental_option("detach", True)
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

    try:
        # 25 英里範圍 URL
        map_bounds = "[[41.8500,-71.5500],[42.3000,-70.9500]]"
        url = (
            f"https://www.google.com/travel/search?"
            f"q=vacation%20rentals%20near%20Gillette%20Stadium%20June%202026&"
            f"hl=en-US&f=ho:1;mv:{map_bounds};"
        )
        
        driver.get(url)
        print("Waiting for properties... Solve CAPTCHA if needed.")
        
        wait = WebDriverWait(driver, 30)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "c-wiz div a")))

        # 滾動加載
        for _ in range(10):
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(2)

        results = []
        # 保持你原本成功的選擇器
        property_links = driver.find_elements(By.CSS_SELECTOR, "c-wiz div a")
        print(f"Found {len(property_links)} potential links. Starting parse...")

        for link_element in property_links:
            try:
                # 核心修正：先拿到父層容器
                parent = link_element.find_element(By.XPATH, "..")
                raw_text = parent.text
                if not raw_text or "$" not in raw_text: 
                    continue

                # --- 提取座標的保險邏輯 ---
                # 我們不強求特定的 div，而是向上找三層，哪一層有 data-coords 就拿哪一層
                lat, lon = "N/A", "N/A"
                temp_element = link_element
                for _ in range(3): # 向上找三層
                    coords = temp_element.get_attribute("data-coords")
                    if coords and "," in coords:
                        lat, lon = coords.split(',')
                        break
                    temp_element = temp_element.find_element(By.XPATH, "..")
                # -----------------------

                lines = raw_text.split('\n')
                name = lines[0]
                price = "N/A"
                miles = "N/A"

                for line in lines:
                    if "$" in line: price = line
                    if "miles" in line.lower(): miles = line

                results.append({
                    "airbnb_name": name,
                    "latitude": lat,
                    "longitude": lon,
                    "miles_from_Gillette": miles,
                    "price_per_night": price,
                    "url": link_element.get_attribute("href")
                })
            except:
                continue

        if results:
            df = pd.DataFrame(results).drop_duplicates(subset=['airbnb_name'])
            df.to_csv("gillette_fifa_with_coords.csv", index=False)
            print(f"Success! {len(df)} properties captured with coordinates.")
            print(df[['airbnb_name', 'latitude', 'longitude']].head())
        else:
            print("Zero properties results. The parsing logic might be blocked.")

    except Exception as e:
        print(f"Script stopped: {e}")

if __name__ == "__main__":
    scrape_gillette_fifa_2026()

Waiting for properties... Solve CAPTCHA if needed.
Found 131 potential links. Starting parse...
Success! 16 properties captured with coordinates.
                                       airbnb_name latitude longitude
0      Courtyard by Marriott Boston Norwood/Canton      N/A       N/A
1                  Four Points by Sheraton Norwood      N/A       N/A
2       Hilton Garden Inn Foxborough Patriot Place      N/A       N/A
3  Residence Inn by Marriott Boston Norwood/Canton      N/A       N/A
4        Holiday Inn Express Boston-Milford by IHG      N/A       N/A


In [76]:
import pandas as pd
from apify_client import ApifyClient

# 1. 確保 Token 正確
client = ApifyClient("apify_api_03hCS4g4bkZHRZkDNEpdl0Ph4T73Nq03iQpB")
actor_name = "compass/crawler-google-places"

# 2. 修正參數名稱 (對應其官方 Schema)
run_input = {
   "searchStringsArray": [
    # 1. 核心住宿類型 (針對你的 Target Categories)
    "Hotels in Foxborough MA",
    "Motels near Gillette Stadium",
    "Vacation home rentals Foxborough",
    "Guest houses near Patriot Place",
    
    # 2. 周邊 15-25 英里的關鍵城鎮 (增加地理覆蓋)
    "Hotels in Mansfield MA",
    "Accommodation in Walpole MA",
    "Hotels in Sharon MA",
    "Wrentham MA hotels and motels",
    "Plainville MA vacation rentals",
    
    # 3. 針對 FIFA 2026 大量人潮的替代方案
    "Extended stay hotels near Foxborough",
    "Bed and Breakfast near Gillette Stadium",
    "Serviced apartments Foxborough MA",
    "Inns and suites near I-95 Massachusetts",
    
    # 4. 廣泛搜尋詞 (觸發地圖聚合點)
    "Places to stay in Foxborough",
    "Lodging near Gillette Stadium"
],
    "locationQuery": "Foxborough, Massachusetts",
    "maxPoints": 500,  # 加大點位數量
    "language": "en",
    "proxyConfig": { "useApifyProxy": True }
}

print(f"啟動 {actor_name}...")

try:
    # 執行 Actor
    run = client.actor(actor_name).call(run_input=run_input)
    
    # 3. 提取結果
    results = []
    for item in client.dataset(run["defaultDatasetId"]).iterate_items():
        results.append({
            "title": item.get("title"),
            "category": item.get("categoryName"),
            "latitude": item.get("location", {}).get("lat"),
            "longitude": item.get("location", {}).get("lng"),
            "address": item.get("address"),
            "stars": item.get("totalScore"),
            "url": item.get("url")
        })

    df = pd.DataFrame(results)
    
    if df.empty:
        print("資料表仍為空，請檢查 Apify Console 的輸出結果。")
    else:
        print(f"成功！抓取到 {len(df)} 筆資料。")
        print(df.head())

except Exception as e:
    print(f"執行出錯: {e}")

啟動 compass/crawler-google-places...


[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> Status: RUNNING, Message: Starting the crawler.
[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> 2026-04-16T23:46:43.266Z ACTOR: Pulling container image of build Mg5SFM40fYCoLTuJO from registry.
[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> 2026-04-16T23:46:43.269Z ACTOR: Creating container.
[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> 2026-04-16T23:46:43.310Z ACTOR: Starting container.
[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> 2026-04-16T23:46:43.310Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> 2026-04-16T23:46:44.281Z INFO  System info {"apifyVersion":"3.5.3","apifyClientVersion":"2.19.0","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v24.14.1"}
[apify.crawler-google-places runId:aQtPHmAxXOBA7bttt] -> 2026-04-16T23:46:44.682Z INFO  📡 Geolocation started
[apify.crawler-google-places runId:aQtPHmAxXOB

成功！抓取到 309 筆資料。
                                         title category   latitude  longitude  \
0   Hilton Garden Inn Foxborough Patriot Place    Hotel  42.091147 -71.268563   
1   Sonesta Select Boston Foxborough Mansfield    Hotel  42.039540 -71.236894   
2    Hampton Inn & Suites Foxborough/Mansfield    Hotel  42.036462 -71.232772   
3       Renaissance Boston Patriot Place Hotel    Hotel  42.090448 -71.268529   
4  Residence Inn by Marriott Boston Foxborough    Hotel  42.044727 -71.238334   

                                     address  stars  \
0        27 Patriot Pl, Foxborough, MA 02035    4.4   
1   35 Foxborough Blvd, Foxborough, MA 02035    4.1   
2    2 Foxborough Blvd, Foxborough, MA 02035    4.6   
3        28 Patriot Pl, Foxborough, MA 02035    4.4   
4  250 Foxborough Blvd, Foxborough, MA 02035    4.3   

                                                                                                                                                url  
0   https://www.

In [81]:
import numpy as np

def calculate_miles(lat1, lon1, lat2, lon2):
    # 地球半徑（英里）
    R = 3958.8 
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# Gillette Stadium 座標
STADIUM_LAT, STADIUM_LON = 42.0909, -71.2643

# 執行計算
df['miles_to_stadium'] = df.apply(
    lambda row: calculate_miles(row['latitude'], row['longitude'], STADIUM_LAT, STADIUM_LON), 
    axis=1
)

# 篩選 25 英里內並排序
df_final = df[df['miles_to_stadium'] <= 25].sort_values(by='miles_to_stadium')

print(f"篩選完成，25英里內共有 {len(df_final)} 個房源。")
df_final.to_csv("new_gillette_25mi_analysis.csv", index=False)

篩選完成，25英里內共有 309 個房源。


In [78]:
#df=pd.read_csv('gillette_25mi_analysis.csv')
# 建立一個清單，放入你想篩選的所有類別
target_categories = ['Hotel', 'Villa', 'House','Motel']

# 使用 .isin() 進行過濾
filtered_df = df_final[df_final['category'].isin(target_categories)]

print(f"符合條件的數量: {len(filtered_df)}")

符合條件的數量: 24


In [79]:
#df=pd.read_csv('gillette_25mi_analysis.csv')
# 建立一個清單，放入你想篩選的所有類別
target_categories = ['Restaurant','Bar & grill','Pizza restaurant','Bagel shop','Shopping mall']

# 使用 .isin() 進行過濾
filtered_df = df_final[df_final['category'].isin(target_categories)]

print(f"符合條件的數量: {len(filtered_df)}")

符合條件的數量: 13


In [80]:
df_final['category']

216                    Stadium
217                     Museum
218           Sportswear store
187                 Sports bar
215                 Restaurant
                ...           
80              Fitness center
82     Physical therapy clinic
288       Electric motor store
262          Paving contractor
118           Firearms academy
Name: category, Length: 309, dtype: object

In [57]:
#airbnb
import pandas as pd
from apify_client import ApifyClient

# 1. 初始化
client = ApifyClient("apify_api_03hCS4g4bkZHRZkDNEpdl0Ph4T73Nq03iQpB")
actor_name = "apify/airbnb-scraper" # 使用官方版本最保險

# 2. 設定精確搜尋參數
run_input = {
    "locationQuery": "Foxborough, Massachusetts", # 建議寫全名
    "checkIn": "2026-06-12",
    "checkOut": "2026-06-14",
    "maxListings": 500,
    "includeReviews": False,
    "currency": "USD",
    "proxyConfiguration": { "useApifyProxy": True }
}

print(f"正在啟動官方 {actor_name} 爬取 FIFA 2026 房源資料...")

try:
    run = client.actor(actor_name).call(run_input=run_input)
    
    results = []
    # 3. 提取 Airbnb 專有的詳細維度
    for item in client.dataset(run["defaultDatasetId"]).iterate_items():
        results.append({
            "name": item.get("name"),
            "price_per_night": item.get("pricing", {}).get("rate", {}).get("amount"),
            "total_price": item.get("pricing", {}).get("total", {}).get("amount"),
            "latitude": item.get("location", {}).get("lat"),
            "longitude": item.get("location", {}).get("lng"),
            "personCapacity": item.get("personCapacity"), # 容納人數
            "bedrooms": item.get("bedrooms"),             # 臥室數
            "beds": item.get("beds"),                     # 床位數
            "stars": item.get("stars"),
            "url": f"https://www.airbnb.com/rooms/{item.get('id')}"
        })

    df = pd.DataFrame(results)
    
    if df.empty:
        print("警告：抓取結果為空。請檢查 Apify Console 的輸入日誌是否有驗證碼報錯。")
    else:
        # 4. 這裡可以接續你的 Haversine 距離計算
        print(f"成功！已抓取 {len(df)} 筆帶有詳細規格的房源資料。")
        df.to_csv("airbnb_fifa_full_details.csv", index=False)
        print(df.head())

except Exception as e:
    print(f"執行時出錯: {e}")

正在啟動官方 apify/airbnb-scraper 爬取 FIFA 2026 房源資料...
執行時出錯: Actor with this name was not found
